In [4]:
import os
import json
from tavily import TavilyClient
from openai import OpenAI
from typing import List, Dict
import csv
from trulens.apps.custom import TruCustomApp
from trulens.core import TruSession
from trulens.apps.custom import instrument
from dotenv import load_dotenv

load_dotenv()


True

In [6]:
session = TruSession()

## Uncomment the following to reset database 
# session.reset_database()

Updating app_name and app_version in apps table: 0it [00:00, ?it/s]
Updating app_id in records table: 0it [00:00, ?it/s]
Updating app_json in apps table: 0it [00:00, ?it/s]


In [7]:
# with open("../../GroundTruths_Dataset -No Multihop No yes or no questions.csv", mode='r', encoding='utf-8') as file:
#     csv_reader = csv.DictReader(file)
#     # Iterate through rows as dictionaries
#     queries = []
#     for row in csv_reader:
#         queries.append(row["query"]) 

with open("../../GroundTruths_Dataset - Sheet1.csv", mode='r', encoding='utf-8') as file:
    csv_reader = csv.DictReader(file)
    # Iterate through rows as dictionaries
    queries = []
    for row in csv_reader:
        queries.append(row["query"]) 

len(queries)

23

In [8]:
tavily_client = TavilyClient(api_key=os.getenv("TAVILYAI_API_KEY"))
llm = OpenAI()

In [9]:
from pinecone.grpc import PineconeGRPC as Pinecone


pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY_500"))
index = pc.Index("openai-test")

In [10]:
embed = llm.embeddings.create


In [11]:
class retriever:
        def __init__(self, embed, index):
             self.embed = embed
             self.index = index
        def get_data(self,query):
            k=5
            embedding=self.embed( model="text-embedding-3-large",input=query).data[0].embedding

            vecs = self.index.query(
            vector=embedding,
            top_k=k,
            includeMetadata=True,
            include_values=True
        )["matches"]
            ids=[] 
            for match in vecs:
                ids.append(match.id)
            data = self.index.fetch(ids)
            docs = []
            for key in data["vectors"]:
                docs.append(data["vectors"][key]["metadata"]["text"])
            filtered_docs = self.__evaluate(docs, query)
            n = k-len(filtered_docs)

            if n==0:
                 return filtered_docs
            else:
                 print(f"{n} chunks will be retrieved from web query")
                 search_docs = self._web_search(query,n)
                 filtered_docs.extend(search_docs)

                 
                 return filtered_docs


        def __evaluate(self,chunks, query):
             res=  llm.chat.completions.create(
                                model="gpt-4o-mini",
                                messages=[
                                    {
                                    "role": "system",
                                    "content": [
                                        {
                                        "text": "Evaluate the relevance between queries and chunks, grading each chunk as \"yes\" if relevant and \"no\" if irrelevant.\n\n# Steps\n\n1. Analyze the given query and each chunk.\n2. Compare the content of the chunk with the context and intent of the query.\n3. Determine the relevance based on logical consistency, contextual alignment, and key theme extraction.\n4. Conclude the evaluation by grading the chunk as \"yes\" if it is relevant to the query, or \"no\" if it is irrelevant.\n\n# Output Format\n\n- A list indicating each chunk's relevance with \"yes\" or \"no\" based on its alignment with the query.\n\n# Examples\n\n- **Query:** \"How do solar panels work?\"\n  - **Chunk 1:** \"The functioning of solar panels involves the conversion of sunlight into electricity.\" \n    - **Relevance:** \"yes\"\n  - **Chunk 2:** \"Solar panels are made from a variety of materials including monocrystalline silicon.\"\n    - **Relevance:** \"yes\"\n  - **Chunk 3:** \"Global oil prices fluctuated significantly last year.\"\n    - **Relevance:** \"no\"\n\n- **Query:** \"What are the health benefits of green tea?\"\n  - **Chunk 1:** \"Green tea contains antioxidants which are beneficial for health.\"\n    - **Relevance:** \"yes\"\n  - **Chunk 2:** \"The manufacturing process of green tea differs from black tea.\"\n    - **Relevance:** \"no\"",
                                        "type": "text"
                                        }
                                    ]
                                    },
                                    {
                                    "role": "user",
                                    "content": [
                                        {
                                        "text": f"Query: {query} \n Chunks:{chunks} ",
                                        "type": "text"
                                        }
                                    ]
                                    }
                                ],
                                response_format={
                                    "type": "json_schema",
                                    "json_schema": {
                                    "name": "evaluated_chunks",
                                    "schema": {
                                        "type": "object",
                                        "required": [
                                        "chunks"
                                        ],
                                        "properties": {
                                        "chunks": {
                                            "type": "array",
                                            "items": {
                                            "type": "object",
                                            "required": [
                                                "chunk",
                                                "evaluation"
                                            ],
                                            "properties": {
                                                "chunk": {
                                                "type": "string",
                                                "description": "A string representing a chunk of data."
                                                },
                                                "evaluation": {
                                                "enum": [
                                                    "yes",
                                                    "no"
                                                ],
                                                "type": "string",
                                                "description": "Evaluation result which can either be 'yes' or 'no'."
                                                }
                                            },
                                            "additionalProperties": False
                                            },
                                            "description": "A list of evaluated chunks."
                                        }
                                        },
                                        "additionalProperties": False
                                    },
                                    "strict": True
                                    }
                                },
                                temperature=0,
                                max_completion_tokens=2049,
                                top_p=1,
                                frequency_penalty=0,
                                presence_penalty=0
                                )
             filtered_docs =[]
             r = json.loads(res.choices[0].message.content)
             for content in r["chunks"]:
                  if content["evaluation"] == "yes":
                       filtered_docs.append(content["chunk"])
             return filtered_docs
        
        def _web_search(self, query, n=5):
             response = tavily_client.search(f"For MOHAP (Ministry of Health in the UAE): {query}",max_results=n)["results"]
             return [item["content"] for item in response]


In [12]:
openai_retreiver = retriever(embed, index)

In [ ]:
# ## Test
# query="what is the the service i use to register as new practicing doctor in the UAE "
# res = openai_retreiver.get_data(query)
# print(res)
# print(type(res))

In [13]:
prompt ="Given the provided context, generate a response that is accurate, concise, and strictly aligned with the information retrieved. Ensure the response does not include hallucinations, speculations, or unsupported claims. The response should be neutral, fact-based, and respectful, especially when addressing sensitive or ambiguous topics. If the context provided is insufficient, clearly state that more information is needed. Prioritize safety and relevance, and avoid generating offensive or harmful content. Please the answer should be in paragraph style and sentences. do not introduce lists and bullet points"

class generator:
    def __init__(self, llm):
        self.llm = llm
    
    def generate(self, query, context):
        formatted_context = "\n".join([str(doc) for doc in context])
        response = self.llm.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": prompt},
        {
            "role": "user",
            "content": query+formatted_context
        }
    ]
)
        return response.choices[0].message

In [14]:
llm = OpenAI(api_key=os.getenv("OPEN_AI_EVAL_KEY"))
gen = generator(llm)


In [15]:
gen.generate(queries[15], openai_retreiver.get_data(queries[15]))

3 chunks will be retrieved from web query


ChatCompletionMessage(content='To obtain a certificate of good standing and professional conduct in the UAE for medical staff, specific requirements vary depending on whether the applicant works in the government or private sector. For those in the private sector, a copy of the medical profession practice card (which is an electronic license issued by MOHAP), a valid passport, and an experience certificate issued from the private medical facility are required. For medical staff in the government sector, a letter of experience from the Department of Human Resources of the Ministry of Health and Prevention or Emirates Health Services, an electronic experience certificate signed by the medical or technical director, and a letter of experience from the employer approved by the outsourcing company are necessary. It is important to note that the certificate will not be issued to trainees, visitors, or those with only an initial license, and the individual must have been licensed by the Minis

In [16]:
class Rag_app:
    def __init__(self,llm,retriever):
        self.retriever = retriever
        self.llm=llm
    @instrument
    def retrieve(self, query: str) -> List[str]:
        """
        Method to handle document retrieval.
        IMPORTANT: The method name 'retrieve' will be used in selectors
        """
        documents = self.retriever.get_data(query)
        return documents
    
    @instrument
    def generate(self, query: str, context: List[str]) -> str:
        """
        Method to handle response generation.
        IMPORTANT: The method name 'generate' will be used in selectors
        """
        formatted_context = "\n".join([str(doc) for doc in context])
        response = self.llm.generate(query ,formatted_context)
        return response
    
    @instrument
    def query(self, question: str) -> Dict:
        """
        Main method that orchestrates the RAG pipeline.
        IMPORTANT: Return keys must match selector paths
        """
        context = self.retrieve(question)
        response = self.generate(question, context)
        
        return response.content

In [17]:
rag_app = Rag_app(generator(llm), openai_retreiver)
# provider = OpenAI(model_engine="gpt-4o", api_key=os.getenv("OPEN_AI_EVAL_KEY"))

import numpy as np
from trulens.core import Feedback
from trulens.core import Select
from trulens.providers.openai import OpenAI
provider = OpenAI()

# Define a groundedness feedback function
f_groundedness = (
    Feedback(
        provider.groundedness_measure_with_cot_reasons, name="Groundedness"
    )
    .on(Select.RecordCalls.retrieve.rets.collect())
    .on_output()
)
# Question/answer relevance between overall question and answer.
f_answer_relevance = (
    Feedback(provider.relevance_with_cot_reasons, name="Answer Relevance")
    .on_input()
    .on_output()
)

# Context relevance between question and each context chunk.
f_context_relevance = (
    Feedback(
        provider.context_relevance_with_cot_reasons, name="Context Relevance"
    )
    .on_input()
    .on(Select.RecordCalls.retrieve.rets[:])
    .aggregate(np.mean)  # choose a different aggregation method if you wish
)

✅ In Groundedness, input source will be set to __record__.app.retrieve.rets.collect() .
✅ In Groundedness, input statement will be set to __record__.main_output or `Select.RecordOutput` .
✅ In Answer Relevance, input prompt will be set to __record__.main_input or `Select.RecordInput` .
✅ In Answer Relevance, input response will be set to __record__.main_output or `Select.RecordOutput` .
✅ In Context Relevance, input question will be set to __record__.main_input or `Select.RecordInput` .
✅ In Context Relevance, input context will be set to __record__.app.retrieve.rets[:] .


In [24]:
from trulens.apps.custom import TruCustomApp
##NAMING CONVENTION eval-{Retriever}-{generator}-{chunksize}-@{k}

tru_rag = TruCustomApp(
    rag_app,
    app_name="Corrective RAG",
    app_version="4o-large_3-500-multihop&Yes-No",
    feedbacks=[f_groundedness, f_answer_relevance, f_context_relevance],
)

In [ ]:
# rag_app.query(queries[19])

In [19]:
from trulens.core.utils.pace import Pace

# Define your desired pacing rate
pace = Pace(marks_per_second=0.5, seconds_per_period=30.0)

In [25]:
queries[16:]

[' I have a medical equipment that is manufactured from animal-based products, What is the condition to renew the license? ',
 'What are the fees to renew the document I get to open a clinic in the UAE? ',
 'What are the required documents to apply for the approval of the service that enables employees to apply and approve their applications but for private entity employees? ',
 'Do healthcare facilities need to fulfill specific requirements regarding elevator installations and what medical staff arrangements are required for operations?',
 'Can pharmaceutical companies export narcotic drugs and what are the validity requirements for such permits?',
 'Are there specific requirements for medical professionals over 60 years old and what documentation is needed for their continued practice?',
 'What restrictions apply to medical advertising on websites and social media, and how does the licensing differ between platforms?']

In [28]:
with tru_rag as recording:
    for eval in queries[15:]:
        print(eval)
        pace.mark()
        rag_app.query(
        eval
    )

What are the requirement documents for the good standing certificate of medical staff in the sector the is fee-exempt for renewal staff licenses?
2 chunks will be retrieved from web query
 I have a medical equipment that is manufactured from animal-based products, What is the condition to renew the license? 
2 chunks will be retrieved from web query
What are the fees to renew the document I get to open a clinic in the UAE? 
2 chunks will be retrieved from web query
What are the required documents to apply for the approval of the service that enables employees to apply and approve their applications but for private entity employees? 
3 chunks will be retrieved from web query
Do healthcare facilities need to fulfill specific requirements regarding elevator installations and what medical staff arrangements are required for operations?
3 chunks will be retrieved from web query
Can pharmaceutical companies export narcotic drugs and what are the validity requirements for such permits?
1 chun

In [22]:
from trulens.dashboard import run_dashboard

run_dashboard(session)

Starting dashboard ...
Dashboard already running at path:   Network URL: http://192.168.1.12:12249



<Popen: returncode: None args: ['streamlit', 'run', '--server.headless=True'...>